# 22 — Prediction Interval / 预测区间

**Chapter 22 — File 3 of 3**

## Summary / 摘要

A prediction interval quantifies uncertainty in individual predictions from a regression model. The interval width equals 1.96 times the standard deviation of residuals, which represents the typical prediction error at any x value (for 95% confidence). Unlike confidence intervals (which bound the mean), prediction intervals account for both estimation uncertainty and inherent data variability. Prediction intervals are always wider than confidence intervals because they include the irreducible noise in the data.

预测区间量化了回归模型中单个预测的不确定性。区间宽度等于残差标准差的1.96倍，这代表任何x值处的典型预测误差（95%置信度）。与置信区间（界定均值）不同，预测区间考虑了估计不确定性和固有数据变异性。预测区间总是比置信区间更宽，因为它们包括数据中的不可约噪声。

## Step 1 — Import Libraries and Prepare Data / 导入库和准备数据

In [ ]:
# Import required libraries
# 导入所需库
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Set random seed for reproducibility
# 设置随机种子以保证可重复性
np.random.seed(42)

## Step 2 — Generate and Fit Data / 生成和拟合数据

In [ ]:
# Generate synthetic data
# 生成合成数据
x = np.arange(0, 10, 0.5)
noise = np.random.normal(0, 2, len(x))
y = 2 * x + 1 + noise

# Fit linear regression
# 拟合线性回归
slope, intercept, r_value, p_value, std_err = linregress(x, y)

# Calculate fitted values and residuals
# 计算拟合值和残差
y_pred = slope * x + intercept
residuals = y - y_pred
residuals_std = np.std(residuals, ddof=1)

# Display model summary
# 显示模型摘要
print(f"Model Summary:")
print(f"  Fitted: y = {slope:.4f}x + {intercept:.4f}")
print(f"  R-squared: {r_value**2:.4f}")
print(f"  Residual std: {residuals_std:.4f}")
print(f"  Sample size: {len(x)}")

## Step 3 — Calculate Prediction Interval / 计算预测区间

In [ ]:
# Prediction interval parameters
# 预测区间参数
confidence_level = 0.95  # 95% confidence / 95%置信
z_critical = 1.96  # Critical value for 95% / 95%的临界值

# Calculate prediction interval margins
# 计算预测区间余量
# For simple regression, prediction margin = z * residual_std
# 对于简单回归，预测余量 = z * 残差_std
prediction_margin = z_critical * residuals_std

# Calculate PI bounds at each observed x
# 计算每个观察x处的PI边界
pi_lower = y_pred - prediction_margin
pi_upper = y_pred + prediction_margin

# Display prediction interval details
# 显示预测区间详情
print(f"\nPrediction Interval Calculation:")
print(f"  Confidence level: {confidence_level*100:.0f}%")
print(f"  Z-critical value: {z_critical}")
print(f"  Residual std: {residuals_std:.4f}")
print(f"  Margin of error: {prediction_margin:.4f}")
print(f"\nPI width at all points: {2*prediction_margin:.4f}")
print(f"\nSample predictions:")
print(f"{'x':>5} | {'y_pred':>8} | {'PI_lower':>8} | {'PI_upper':>8}")
print("-" * 36)
for i in range(min(5, len(x))):
    print(f"{x[i]:5.1f} | {y_pred[i]:8.2f} | {pi_lower[i]:8.2f} | {pi_upper[i]:8.2f}")

## Step 4 — Generate Smooth Prediction Curves / 生成光滑预测曲线

In [ ]:
# Create smooth prediction lines for visualization
# 为可视化创建光滑预测线
x_smooth = np.linspace(x.min(), x.max(), 100)

# Predict on smooth x
# 在光滑x上预测
y_smooth_pred = slope * x_smooth + intercept

# Prediction interval bounds on smooth x
# 光滑x上的预测区间边界
pi_smooth_lower = y_smooth_pred - prediction_margin
pi_smooth_upper = y_smooth_pred + prediction_margin

# Calculate confidence interval for reference
# 计算参考的置信区间
# Standard error of the mean prediction
# 均值预测的标准误差
x_mean = np.mean(x)
se_mean = np.sqrt(np.sum((x - x_mean)**2))  # Used for scaling / 用于缩放
se_fit = residuals_std * np.sqrt(1/len(x) + (x_smooth - x_mean)**2 / np.sum((x - x_mean)**2))
ci_margin = z_critical * se_fit
ci_lower = y_smooth_pred - ci_margin
ci_upper = y_smooth_pred + ci_margin

print(f"\nSmooth curves generated for {len(x_smooth)} points")

## Step 5 — Visualize Data with Prediction Intervals / 用预测区间可视化数据

In [ ]:
# Create comprehensive visualization
# 创建综合可视化
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Prediction interval with data
# 图1: 预测区间与数据
# Shade the prediction interval region
# 给预测区间区域着色
axes[0].fill_between(x_smooth, pi_smooth_lower, pi_smooth_upper, 
                     alpha=0.2, color='red', label='95% Prediction Interval')

# Shade the confidence interval region
# 给置信区间区域着色
axes[0].fill_between(x_smooth, ci_lower, ci_upper, 
                     alpha=0.3, color='blue', label='95% Confidence Interval')

# Plot fitted line
# 绘制拟合线
axes[0].plot(x_smooth, y_smooth_pred, 'g-', linewidth=2, label='Fitted line')

# Plot PI and CI bounds
# 绘制PI和CI边界
axes[0].plot(x_smooth, pi_smooth_lower, 'r--', linewidth=1.5, alpha=0.7)
axes[0].plot(x_smooth, pi_smooth_upper, 'r--', linewidth=1.5, alpha=0.7)
axes[0].plot(x_smooth, ci_lower, 'b--', linewidth=1.5, alpha=0.7)
axes[0].plot(x_smooth, ci_upper, 'b--', linewidth=1.5, alpha=0.7)

# Plot observed data
# 绘制观察数据
axes[0].scatter(x, y, s=50, color='black', alpha=0.6, label='Observed data')

axes[0].set_xlabel('X', fontsize=12)
axes[0].set_ylabel('Y', fontsize=12)
axes[0].set_title('Linear Regression with Prediction Interval', fontsize=14)
axes[0].legend(fontsize=10, loc='upper left')
axes[0].grid(True, alpha=0.3)

# Plot 2: Error bar representation
# 图2: 误差条表示
# Select subset of x for clarity
# 选择x的子集以便清晰
x_subset = x[::3]  # Every 3rd point / 每3个点
y_pred_subset = y_pred[::3]
pi_lower_subset = pi_lower[::3]
pi_upper_subset = pi_upper[::3]

# Calculate errors for error bar
# 计算误差条的误差
errors_pi = np.array([y_pred_subset - pi_lower_subset, pi_upper_subset - y_pred_subset])

axes[1].errorbar(x_subset, y_pred_subset, yerr=errors_pi, fmt='o', 
                markersize=8, capsize=10, capthick=2, linewidth=2, 
                color='red', ecolor='red', label='Prediction Interval')
axes[1].scatter(x, y, s=50, color='black', alpha=0.6, label='Observed data')
axes[1].plot(x_smooth, y_smooth_pred, 'g-', linewidth=2, label='Fitted line')

axes[1].set_xlabel('X', fontsize=12)
axes[1].set_ylabel('Y', fontsize=12)
axes[1].set_title('Prediction Interval as Error Bars', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6 — Compare Intervals and Analyze Width / 比较区间和分析宽度

In [ ]:
# Calculate average widths
# 计算平均宽度
pi_width_avg = np.mean(pi_upper - pi_lower)
ci_width_avg = np.mean(ci_upper - ci_lower)

# Compare intervals
# 比较区间
print(f"\nInterval Comparison:")
print(f"  Prediction Interval (PI) average width: {pi_width_avg:.4f}")
print(f"  Confidence Interval (CI) average width: {ci_width_avg:.4f}")
print(f"  PI/CI ratio: {pi_width_avg/ci_width_avg:.2f}x wider")
print(f"\nReason: PI includes both estimation uncertainty AND inherent data variability")
print(f"  Residual std (data variability): {residuals_std:.4f}")
print(f"  Standard error at center (estimation uncertainty): {se_fit[len(se_fit)//2]:.4f}")

In [ ]:
## Learning Notes / 学习笔记

- **Statistical Concept**: Prediction intervals are wider than confidence intervals because they account for two sources of uncertainty: (1) uncertainty in estimating the regression line (standard error) and (2) inherent variability of individual observations around the true line (residual std). The prediction margin = z × residual_std captures both components. As sample size increases, the standard error decreases, but the residual std remains constant, so PI width becomes increasingly dominated by residual variability.
  
  **统计概念**: 预测区间比置信区间更宽，因为它们考虑了两种不确定性来源：(1)估计回归线的不确定性（标准误差），和(2)单个观察围绕真实线的固有变异性（残差标准差）。预测余量 = z × 残差_std捕获两个组成部分。随着样本大小增加，标准误差减少，但残差标准差保持不变，因此PI宽度变得越来越由残差变异性主导。

- **ML Application**: In production ML systems, prediction intervals set operational bounds for individual predictions. Wider intervals indicate higher uncertainty and may trigger manual review or alternative processing. For business forecasting, PI widths inform confidence in decisions; narrow intervals allow aggressive strategies while wide intervals require conservative approaches. Monitoring PI width changes over time detects model degradation (increasing residual std suggests distributional shift).
  
  **ML应用**: 在生产ML系统中，预测区间为单个预测设置操作边界。更宽的区间表示更高的不确定性，可能会触发手动审查或替代处理。对于业务预测，PI宽度告知决策的信心；窄区间允许激进策略，而宽区间需要保守方法。随时间监控PI宽度变化可检测模型退化（残差标准差增加表示分布偏移）。

➡️ **Next**: `../chapter_23/01_rank_data.ipynb`

## Complete Code / 完整代码一览

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

np.random.seed(42)
x = np.arange(0, 10, 0.5)
noise = np.random.normal(0, 2, len(x))
y = 2 * x + 1 + noise

slope, intercept, r_value, p_value, std_err = linregress(x, y)
y_pred = slope * x + intercept
residuals = y - y_pred
residuals_std = np.std(residuals, ddof=1)

z_critical = 1.96
prediction_margin = z_critical * residuals_std

pi_lower = y_pred - prediction_margin
pi_upper = y_pred + prediction_margin

print(f"Prediction Interval width: {2*prediction_margin:.4f}")
print(f"Sample PI at x=0: [{pi_lower[0]:.2f}, {pi_upper[0]:.2f}]")